# Suite PRV — Provenance: the records of the transactions a store executes

A service needs to know who changed what, when and from where ([CAP-134: transactions recorded with provenance](../../../mbse-basis/docs/capabilities/CAP-134.md);
FRAMEWORK.md, Resolved, Provenance). `Provenance`, a module installed before use (`store.transaction(Provenance.install)`),
holds a record of each executed transaction (`Provenance.OfTransaction`, keyed by its `uid`: its `when` and `where`), the
session it was executed in (`Provenance.OfSession`, keyed by its `id`, through `Provenance.InSession`) and the session's
identity (`Provenance.OfIdentity`, keyed by its `name`, through `Provenance.Who`). `Provenance.extend(txn, schemas)`
gives the reference object schemas an app chooses anonymous adjacencies to `Provenance.CreatedBy` and
`Provenance.ModifiedBy`; the executor writes the records at commit, within the transaction, from what its expression
gives (`Mutations.OfTransaction`'s `when`, `session` and `where`), so that a replay writes nothing twice. An imperative
scope's record has placeholder data (Decided, 3; tech debt).

As in Python, with the language's differences (docs/EQUIVALENCE.md): builders and the store are `Proxy` objects typed
loosely, so callbacks are annotated `(t: any)`; a frame is a `Map`, plain mappings are `Map`s, and what Python reads as
None is `null` or `undefined`.

In [ ]:
import { Builders as B, JSON as Json, Plain, Provenance, References } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises } from "./support.js";

/** work.Feature, owned by the singleton board by key; work.Comment, owned by the board, which gives each its ID, and
 * about a feature. */
function work(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("work/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Board: S.OfObject, Holds: S.OfRelation, Posted: S.OfRelation, Feature: S.OfObject,
      Comment: S.OfObject, About: S.OfRelation })
    .spec([
      (_: any) => _.Holds.name("work.Holds").links([(t: any) => t.name("board"), (t: any) => t.name("feature")])
        .properties({ key: S.OfStr }).uniques([["board", "key"], ["feature"]]),
      (_: any) => _.Posted.name("work.Posted").links([(t: any) => t.name("board"), (t: any) => t.name("comment")])
        .properties([(t: any) => t.name("id").schema([S.OfStr]).pattern("C-(\\d+)")]).uniques([["board", "id"],
          ["comment"]]),
      (_: any) => _.Board.name("work.Board").ref(true).singleton("work.TheBoard").adjacencies({
        features: (a: any) => a.relation([_.Holds]).me("board"),
        comments: (a: any) => a.relation([_.Posted]).me("board") }),
      (_: any) => _.Feature.name("work.Feature").ref(true).properties({ title: S.OfStr }).adjacencies([
        (t: any) => t.name("board").relation([_.Holds]).me("feature").owner(true)]),
      (_: any) => _.About.name("work.About").links([(t: any) => t.name("comment"), (t: any) => t.name("feature")]),
      (_: any) => _.Comment.name("work.Comment").ref(true).properties({ text: S.OfStr }).adjacencies([
        (t: any) => t.name("board").relation([_.Posted]).me("comment").owner(true),
        (t: any) => t.name("about").relation([_.About]).me("comment")]),
    ])]))]);
}

/** A store holding work's schemas and Provenance, work.Comment extended, and the feature `chips`. */
function made(): any {
  const store: any = new B.Store();
  store.transaction(work);
  store.transaction(Provenance.install);
  store.transaction((t: any) => Provenance.extend(t, [t.work.Comment]));
  store.transaction((t: any) => t.uid("feature-1").when("2026-10-26T14:00:00Z").mutations([(mu: any) => mu.upsert(
    (u: any) => u.exists([(p: any) => p.symbols({ b: store.work.Board, f: store.work.Feature }).spec([
      (_: any) => _.b.features({ chips: (e: any) => e.feature(_.f) }), (_: any) => _.f.title("Chips")])]))]));
  return store;
}

/** The expression of a transaction posting a comment about `chips`, from a session, as a service sets it. */
function post(store: any, uid: string, when: string, text: string, session: unknown = null, where: unknown = null): any {
  return store.Mutations.OfTransaction((t: any) => {
    t.uid(uid).when(when);
    if (session !== null) t.session(session);
    if (where !== null) t.where(where);
    return t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
      .symbols({ topic: References.of(store, entry(store.singleton("work.TheBoard"), "features", "key", "chips").feature),
        c: store.work.Comment })
      .spec([(_: any) => _.c.text(text).about([_.topic])])]))]);
  });
}

/** The expression of a transaction writing a comment's text, given its reference. */
function edit(store: any, comment: unknown, uid: string, text: string): any {
  return store.Mutations.OfTransaction((t: any) => t.uid(uid).when("2026-10-26T15:00:00Z")
    .session((s: any) => s.id("tab-7f3a").who("ada")).mutations([(mu: any) => mu.upsert((u: any) => u.exists([
      (p: any) => p.symbols({ c: References.of(store, comment) }).spec([(_: any) => _.c.text(text)])]))]));
}

/** The uids of the records the store holds, in the order they were made. */
function uids(store: any): string[] {
  return [...store.singleton("Provenance.Transactions").transactions].map((held: any) => String(held.uid));
}

function record(store: any, uid: string): any {
  return entry(store.singleton("Provenance.Transactions"), "transactions", "uid", uid).transaction;
}

/** The uids of the records whose adjacency `created` or `modified` links `target`. */
function by(store: any, adjacency: string, target: unknown): string[] {
  return [...store.singleton("Provenance.Transactions").transactions]
    .filter((held: any) => [...held.transaction[adjacency]].some((linked: any) => linked.object === target))
    .map((held: any) => String(held.uid));
}

/** A directory singleton's entry keyed by `value`. */
function entry(singleton: any, adjacency: string, key: string, value: string): any {
  return [...singleton[adjacency]].find((held: any) => String(held[key]) === value);
}

function keys(singleton: any, adjacency: string, key: string): string[] {
  return [...singleton[adjacency]].map((held: any) => String(held[key]));
}

const store = made();
const board = store.singleton("work.TheBoard");
const chips = entry(board, "features", "key", "chips").feature;
const UUID = /^[0-9a-f]{8}-[0-9a-f]{4}-4[0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}$/;
let first: any;
let comment: any;
let sessions: any;
let session: any;
let identities: any;
let ada: any;

## PRV-01 · Provenance.install: its schemas and directories, run again writing nothing; installing it, and a store without it, record nothing

In [ ]:
{
  const names = ["Provenance.OfWhere", "Provenance.InSessions", "Provenance.SessionDirectory", "Provenance.OfSession",
    "Provenance.InSession", "Provenance.InTransactions", "Provenance.TransactionDirectory",
    "Provenance.OfTransaction", "Provenance.InIdentities", "Provenance.IdentityDirectory", "Provenance.OfIdentity",
    "Provenance.Who", "Provenance.CreatedBy", "Provenance.ModifiedBy"];
  const held = new Map(names.map((name) => [name, store.registered(name)]));
  const [OfTransaction, OfSession] = [held.get("Provenance.OfTransaction"), held.get("Provenance.OfSession")];
  assert(equal([...OfTransaction.properties.keys()], ["when", "where"])
    && OfTransaction.properties.get("where").schema === held.get("Provenance.OfWhere"));
  assert(equal([...OfTransaction.adjacencies.keys()], ["directory", "session", "created", "modified"]));
  assert(equal([...OfSession.properties.keys()], ["started", "locale", "device"])
    && equal([...OfSession.adjacencies.keys()], ["directory", "transactions", "who"]));
  assert(equal([...held.get("Provenance.OfIdentity").adjacencies.keys()], ["directory", "sessions"]));
  assert(equal([...held.get("Provenance.OfWhere").properties.keys()], ["latitude", "longitude"]) && !held.get("Provenance.OfWhere").ref);
  for (const singleton of ["Provenance.Transactions", "Provenance.Sessions", "Provenance.Identities"]) {
    assert(store.singleton(singleton) !== null);
  }
  for (const [relation, unique] of [["Provenance.InSession", "session"], ["Provenance.Who", "identity"],
    ["Provenance.CreatedBy", "transaction"], ["Provenance.ModifiedBy", "transaction"]]) {
    const uniques = [...held.get(relation).uniques].map((set: any) => [...set]); // the other link determines it
    assert(equal(uniques, [[unique]]));
  }
  assert(equal(uids(store), ["feature-1"])); // neither the install nor the extension, which write schemas alone
  const before = new Map((store.names() as string[]).map((name) => [name, store.registered(name)]));
  store.transaction(Provenance.install); // each found by its name
  assert(store.names().length === before.size && (store.names() as string[]).every((name) => store.registered(name) === before.get(name)));
  assert(equal(uids(store), ["feature-1"]));
  const plain: any = new B.Store(); // no Provenance: nothing recorded, nothing changes
  plain.transaction(work);
  plain.transaction((t: any) => t.uid("feature-1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ b: plain.work.Board, f: plain.work.Feature }).spec([
      (_: any) => _.b.features({ chips: (e: any) => e.feature(_.f) }), (_: any) => _.f.title("Chips")])]))]));
  assert(!(plain.names() as string[]).some((name) => name.startsWith("Provenance.")));
  assert(String(entry(plain.singleton("work.TheBoard"), "features", "key", "chips").feature.title) === "Chips");
}

## PRV-02 · Provenance.extend: anonymous adjacencies to CreatedBy and ModifiedBy through `object`, no member; run again, writing nothing; refused before Provenance is installed, or for what is not a reference object schema

In [ ]:
{
  const [Comment, Feature] = [store.registered("work.Comment"), store.registered("work.Feature")];
  const added = Comment.anonymous.map((a: any) => [a.relation.name, a.me, a.name]);
  assert(equal(added, [["Provenance.CreatedBy", "object", ""], ["Provenance.ModifiedBy", "object", ""]]));
  assert(equal([...Comment.adjacencies.keys()], ["board", "about"]) && Feature.anonymous.length === 0); // no member; Feature not extended
  store.transaction((t: any) => Provenance.extend(t, [Comment])); // a schema object, or its path: found, nothing written
  assert(equal(Comment.anonymous.map((a: any) => [a.relation.name, a.me]), added.map(([r, m]: any) => [r, m])));
  const bare: any = new B.Store();
  bare.transaction(work);
  raises(LookupError, () => bare.transaction((t: any) => Provenance.extend(t, [t.work.Comment])),
    "Provenance.extend: the store has no Provenance installed; install it first, store.transaction(Provenance.install)");
  for (const [wrong, shown] of [[(t: any) => t.Provenance.OfWhere, "'Provenance.OfWhere'"], [(_t: any) => "work.Comment", "'work.Comment'"]] as const) {
    raises(TypeError, () => store.transaction((t: any) => Provenance.extend(t, [wrong(t)])),
      `Provenance.extend takes reference object schemas: ${shown} is not one`);
  }
}

## PRV-03 · A transaction with `when`, `session` and `where`: its record, its session and identity made, InSession, Who, and CreatedBy for the comment it creates

In [ ]:
first = post(store, "6f1c4e2a-0000-4000-8000-000000000007", "2026-10-26T14:30:05Z", "Should chips show adjacencies?",
  (s: any) => s.id("tab-7f3a").who("ada").locale("en-GB"), (w: any) => w.latitude(51.5).longitude(-0.125));
comment = store.transaction(first)[0].get("c");
assert(equal(uids(store), ["feature-1", "6f1c4e2a-0000-4000-8000-000000000007"]));
{
  const made_ = record(store, "6f1c4e2a-0000-4000-8000-000000000007");
  assert(String(made_.when) === "2026-10-26T14:30:05Z");
  const where = store.member(made_, "where");
  assert(store.member(where, "latitude") === 51.5 && store.member(where, "longitude") === -0.125);
  sessions = store.singleton("Provenance.Sessions");
  assert(equal(keys(sessions, "sessions", "id"), ["unknown", "tab-7f3a"])); // feature-1's, which named none; tab-7f3a
  session = entry(sessions, "sessions", "id", "tab-7f3a").session;
  assert(String(session.started) === "2026-10-26T14:30:05Z" && String(session.locale) === "en-GB"
    && store.member(session, "device") == null);
  assert(equal([...made_.session].map((held: any) => held.session), [session])); // InSession
  identities = store.singleton("Provenance.Identities");
  assert(equal(keys(identities, "identities", "name"), ["unknown", "ada"])); // feature-1's, which named no session; ada's
  ada = entry(identities, "identities", "name", "ada").identity;
  assert(equal([...session.who].map((held: any) => held.identity), [ada]) && equal([...ada.sessions].map((held: any) => held.session), [session]));
  assert(equal(by(store, "created", comment), ["6f1c4e2a-0000-4000-8000-000000000007"]) && equal(by(store, "modified", comment), []));
  assert(String(entry(board, "comments", "id", "C-1").comment.text) === "Should chips show adjacencies?"); // its directory gave it its ID
}

## PRV-04 · A later write gives ModifiedBy, replaced by the next: the tail; the session found by its id, another session of the identity found by its name

In [ ]:
{
  store.transaction(edit(store, comment, "edit-2", "Should chips show their adjacencies?"));
  assert(equal(by(store, "created", comment), ["6f1c4e2a-0000-4000-8000-000000000007"]) && equal(by(store, "modified", comment), ["edit-2"]));
  store.transaction(edit(store, comment, "edit-3", "Chips should show their adjacencies."));
  assert(equal(by(store, "modified", comment), ["edit-3"]) && equal(uids(store).slice(-2), ["edit-2", "edit-3"])); // edit-2's record stays
  assert(equal(keys(sessions, "sessions", "id"), ["unknown", "tab-7f3a"]) && [...session.transactions].length === 3); // found by its id
  assert(String(record(store, "edit-2").when) === "2026-10-26T15:00:00Z" && store.member(record(store, "edit-2"), "where") == null);
  store.transaction(edit(store, comment, "edit-3", "Again.")); // another write under the same uid
  assert(equal(by(store, "modified", comment), ["edit-3"]) && uids(store).filter((uid) => uid === "edit-3").length === 1); // its uid's record, found
  const phone = post(store, "phone-1", "2026-10-26T16:00:00Z", "From my phone.", (s: any) => s.id("tab-9b2c").who("ada").device(
    "phone"));
  store.transaction(phone);
  assert(equal(keys(sessions, "sessions", "id"), ["unknown", "tab-7f3a", "tab-9b2c"]) && equal(keys(identities, "identities", "name"), [
    "unknown", "ada"]));
  const other = entry(sessions, "sessions", "id", "tab-9b2c").session;
  assert(store.member(other, "locale") == null && String(other.device) === "phone" && equal([...other.who].map((e: any) => e.identity), [ada]));
}

## PRV-05 · Objects of schemas not extended are not linked; a transaction that writes nothing records nothing; one that creates and writes an object in it, CreatedBy alone

In [ ]:
{
  store.transaction((t: any) => t.uid("retitle").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ f: References.of(store, chips) }).spec([(_: any) => _.f.title("Chips, again")])]))]));
  assert(uids(store).includes("retitle") && [...record(store, "retitle").created].length === 0
    && [...record(store, "retitle").modified].length === 0);
  const count = uids(store).length;
  assert(store.transaction((t: any) => t.uid("read").mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ c: store.work.Comment })]))])).length !== 0); // a map: nothing written
  assert(uids(store).length === count);
  const both = store.transaction((t: any) => t.uid("both").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ c: store.work.Comment }).spec([(_: any) => _.c.text("new")])]).then([(n: any) => n.update((d: any) => d.apply([
    (p: any) => p.symbols({ c: store.work.Comment }).spec([(_: any) => _.c.text("newer")])]))]))]))[0].get("c");
  assert(equal(by(store, "created", both), ["both"]) && equal(by(store, "modified", both), []));
}

## PRV-06 · Replaying the same expression writes nothing twice; replayed into another store, the journal makes the same records

In [ ]:
{
  const journal = [first, edit(store, comment, "edit-2", "Should chips show their adjacencies?")]
    .map((expression) => Json.dumps(Plain.ToPlain(store).Transaction(expression))); // the expressions, as a journal holds them
  const again = made();
  const copy = again.transaction(Plain.FromPlain(again).Transaction(Json.loads(journal[0]!)))[0].get("c");
  const directories = ["Provenance.Transactions", "Provenance.Sessions", "Provenance.Identities", "work.TheBoard"]
    .map((name) => again.singleton(name));
  const sizes = (): number[] => [...directories.map((d: any, i: number) => [...d[["transactions", "sessions", "identities",
    "comments"][i]!]].length), [...copy.about].length];
  const before = sizes();
  assert(again.transaction(Plain.FromPlain(again).Transaction(Json.loads(journal[0]!)))[0].get("c") === copy); // replayed: found again
  assert(equal(sizes(), before) && equal(by(again, "created", copy), ["6f1c4e2a-0000-4000-8000-000000000007"])); // nothing twice
  again.transaction(Plain.FromPlain(again).Transaction(Json.loads(journal[1]!)));
  assert(equal(by(again, "created", copy), ["6f1c4e2a-0000-4000-8000-000000000007"]) && equal(by(again, "modified", copy), ["edit-2"]));
  assert(String(record(again, "6f1c4e2a-0000-4000-8000-000000000007").when) === "2026-10-26T14:30:05Z"); // the same records
  assert(equal(keys(again.singleton("Provenance.Sessions"), "sessions", "id"), ["unknown", "tab-7f3a"]));
  assert(String(entry(again.singleton("Provenance.Sessions"), "sessions", "id", "tab-7f3a").session.started)
    === "2026-10-26T14:30:05Z");
}

## PRV-07 · The new transaction members on the wire: `when`, `session` and `where`, written and read back; left out where unused

In [ ]:
{
  const written = Plain.ToPlain(store).Transaction(first) as Map<string, unknown>;
  assert(equal(new Map(["uid", "when", "session", "where"].map((k) => [k, written.get(k)])), map({
    uid: "6f1c4e2a-0000-4000-8000-000000000007", when: "2026-10-26T14:30:05Z",
    session: { id: "tab-7f3a", who: "ada", locale: "en-GB" }, where: { latitude: 51.5, longitude: -0.125 } })));
  assert(equal(Plain.ToPlain(store).Transaction(Plain.FromPlain(store).Transaction(written)), written));
  const text = Json.dumps(written); // as JSON
  assert(equal(Plain.ToPlain(store).Transaction(Plain.FromPlain(store).Transaction(Json.loads(text))), written));
  const bare = store.Mutations.OfTransaction((t: any) => t.uid("x"));
  assert(equal(Plain.ToPlain(store).Transaction(bare), map({ uid: "x" }))); // none of them: as before
  const Session = store.registered("Mutations.OfTransaction.Session");
  assert(equal([...Session.properties.keys()], ["id", "who", "locale", "device"]) && !Session.ref);
  assert(equal([...store.registered("Mutations.OfTransaction.Where").properties.keys()], ["latitude", "longitude"]));
  assert(equal([...store.registered("Mutations.OfTransaction").properties.keys()], ["uid", "when", "session", "where"]));
}

## PRV-08 · An imperative scope, and an expression naming no session or time: placeholder data, a uid the store makes, `when` at commit, the session and identity `unknown`

In [ ]:
{
  let loose: any;
  store.transaction(() => {
    loose = store.work.Comment((c: any) => c.text("by hand").about([chips]));
    store.work.Comment(loose, (c: any) => c.text("by hand, edited")); // created and written in it: CreatedBy alone
  });
  const created = by(store, "created", loose);
  assert(created.length === 1 && UUID.test(created[0]!) && equal(by(store, "modified", loose), []));
  const placeholder = record(store, created[0]!);
  assert(/^\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ$/.test(String(placeholder.when)));
  const [held] = [...placeholder.session];
  assert(equal([...held.session.directory].map((e: any) => String(e.id)), ["unknown"]));
  assert(equal([...[...held.session.who][0].identity.directory].map((e: any) => String(e.name)), ["unknown"]));
  store.transaction(edit(store, loose, "edit-loose", "by a service"));
  assert(equal(by(store, "modified", loose), ["edit-loose"]));
  store.transaction((t: any) => t.mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p.symbols({
    c: References.of(store, loose) }).spec([(_: any) => _.c.text("no uid, no when")])]))]));
  const [latest] = by(store, "modified", loose);
  assert(latest !== "edit-loose" && latest !== created[0] && /^\d{4}-.*Z$/.test(String(record(store, latest!).when)));
  assert(equal([...[...record(store, latest!).session][0].session.directory].map((e: any) => String(e.id)), ["unknown"]));
}

## PRV-09 · Reading an object's records from the record side: a map from a symbol given the object's reference, the record, its session and identity

In [ ]:
{
  const frames = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ c: References.of(store, comment), t: store.Provenance.OfTransaction, i: store.Provenance.OfIdentity,
      s: store.Provenance.OfSession })
    .spec([(_: any) => _.t.created([_.c]), (_: any) => _.t.session([_.s]), (_: any) => _.s.who([_.i])])]))]));
  assert(frames.length === 1); // one frame: t binds the record (its `when`), i the identity (its `name`), s the session
  const found = frames[0];
  assert(String(found.get("t").when) === "2026-10-26T14:30:05Z" && String(found.get("s").locale) === "en-GB");
  assert(equal([...found.get("i").directory].map((held: any) => String(held.name)), ["ada"]));
  const latest = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ c: References.of(store, comment), t: store.Provenance.OfTransaction })
    .spec([(_: any) => _.t.modified([_.c])])]))]));
  assert(equal(latest.map((frame: any) => [...frame.get("t").directory].map((held: any) => String(held.uid))), [["edit-3"]])); // the tail
}

## PRV-10 · What is refused, the transaction with it: a session with no id; a session of another identity

In [ ]:
{
  const count = uids(store).length;
  const nameless = post(store, "nameless", "2026-10-26T17:00:00Z", "No id.", (s: any) => s.who("ada"));
  raises(ValueError, () => store.transaction(nameless),
    "a transaction's session has no id: a service gives it, and who, before executing it");
  const stolen = post(store, "stolen", "2026-10-26T17:00:00Z", "Not mine.", (s: any) => s.id("tab-7f3a").who("eve"));
  raises(ValueError, () => store.transaction(stolen),
    "session 'tab-7f3a' is another identity's, not 'eve''s: a session has one identity (Provenance.Who)");
  assert(uids(store).length === count && ![...board.comments].map((e: any) => String(e.comment.text)).includes("Not mine.")); // nothing written
  assert(equal(keys(identities, "identities", "name"), ["unknown", "ada"]));
}

## PRV-11 · Reading a store's snapshot back restores its records and records nothing of its own, read alone or within a transaction: a restart adds no record

In [ ]:
{
  const source = made();
  source.transaction(post(source, "post-1", "2026-10-26T15:00:00Z", "Restart me", (s: any) => s.id("tab-9").who("ada")));
  const text = Json.ToJSON(source).Snapshot(); // as a restart writes it, and reads it
  for (const restoring of ["alone", "within"]) {
    const restored: any = new B.Store();
    restored.transaction(work);
    restored.transaction(Provenance.install);
    if (restoring === "alone") Json.FromJSON(restored).Snapshot(text);
    else restored.transaction(() => Json.FromJSON(restored).Snapshot(text)); // its writes are the snapshot's, not its own
    const twin = [...restored.singleton("work.TheBoard").comments].map((held: any) => held.comment)
      .find((comment: any) => String(comment.text) === "Restart me");
    assert(equal(by(restored, "created", twin), ["post-1"])); // one creating transaction, restored
    assert(equal(uids(restored), uids(source))); // the records restored, none added
    assert(Json.ToJSON(restored).Snapshot() === text); // restarted again: the same snapshot
  }
}